# 0517 Edge Prior Results Analysis

目标：
- 先审 pipeline 有没有泄露、时间穿越、mask 失效或评估逻辑问题。
- 再看 `G0_hsbm`、`G7_realistic_mix`、`G7_db` 的实验记录和结果，按好/坏/离谱分别解释原因。
- 最后和 `docs/index.html` 里的论文结果做对比。

数据快照：
- 运行根目录：`/local/lzd/plurel_runtime/edge_prior_main_20260517`
- 已完成：`G0_hsbm`、`G7_realistic_mix`
- 已完成：`G7_db`（与 `G0_hsbm`、`G7_realistic_mix` 一起构成完整三组结果）


In [ ]:
from __future__ import annotations

import html as html_lib
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)
plt.style.use("seaborn-v0_8-whitegrid")

REPO_ROOT = Path("/home/lzd/workspace/myplurel/mysyn")
RUNTIME_ROOT = Path("/local/lzd/plurel_runtime/edge_prior_main_20260517")
EXP_ROOT = RUNTIME_ROOT / "rt_main_32x2_steps4001_eval400"
MANIFEST_PATH = RUNTIME_ROOT / "experiments/g0_g7_g7db_paired_n34_seed30000/manifest.json"
DOCS_HTML = REPO_ROOT / "docs/index.html"
REAL_TOPO_PATH = REPO_ROOT / "out/topology_stats/relbench_stats_stable/summary.json"
COHORT_ORDER = ["G0_hsbm", "G7_realistic_mix", "G7_db"]

TASKS = [
    ("rel-amazon", "user-churn", "auc"),
    ("rel-hm", "user-churn", "auc"),
    ("rel-stack", "user-badge", "auc"),
    ("rel-stack", "user-engagement", "auc"),
    ("rel-amazon", "item-churn", "auc"),
    ("rel-avito", "user-visits", "auc"),
    ("rel-avito", "user-clicks", "auc"),
    ("rel-trial", "study-outcome", "auc"),
    ("rel-f1", "driver-dnf", "auc"),
    ("rel-f1", "driver-top3", "auc"),
    ("rel-hm", "item-sales", "r2"),
    ("rel-amazon", "user-ltv", "r2"),
    ("rel-amazon", "item-ltv", "r2"),
    ("rel-stack", "post-votes", "r2"),
    ("rel-trial", "site-success", "r2"),
    ("rel-trial", "study-adverse", "r2"),
    ("rel-f1", "driver-position", "r2"),
    ("rel-avito", "ad-ctr", "r2"),
]
TASK_KIND = {(db_name, table_name): metric_kind for db_name, table_name, metric_kind in TASKS}
AUC_TASKS = [
    (db_name, table_name) for db_name, table_name, metric_kind in TASKS if metric_kind == "auc"
]
R2_TASKS = [
    (db_name, table_name) for db_name, table_name, metric_kind in TASKS if metric_kind == "r2"
]
TOPO_STATS = [
    "fanout_gini",
    "fanout_ks_to_poisson",
    "fanout_ks_to_powerlaw",
    "isolated_parent_rate",
    "degree_assortativity",
    "powerlaw_gamma",
    "temporal_growth_alpha",
]


def metrics_path_for(cohort: str) -> Path:
    candidates = sorted((EXP_ROOT / cohort).glob("*/metrics.jsonl"))
    if not candidates:
        raise FileNotFoundError(f"Missing metrics.jsonl for cohort {cohort!r}")
    return candidates[0]


def load_metrics(cohort: str) -> pd.DataFrame:
    path = metrics_path_for(cohort)
    with path.open(encoding="utf-8") as handle:
        rows = [json.loads(line) for line in handle if line.strip()]
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df["task_key"] = list(zip(df["db_name"], df["table_name"]))
    df["cohort"] = cohort
    df["metrics_path"] = str(path)
    return df


def load_manifest(manifest_path: Path) -> pd.DataFrame:
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    rows = []
    for cohort in manifest["cohorts"]:
        rows.append(
            {
                "cohort": cohort["cohort"],
                "description": cohort["description"],
                "config_hash": cohort["config_hash"],
                "assignment_strategy": cohort["config"]["edge_prior_assignment_strategy"],
                "topology_priors": ", ".join(cohort["config"]["topology_prior_choices"]),
                "num_dbs": cohort["num_dbs"],
            }
        )
    return pd.DataFrame(rows)


def build_run_status(
    manifest_df: pd.DataFrame, metrics_by_cohort: dict[str, pd.DataFrame]
) -> pd.DataFrame:
    rows = []
    for row in manifest_df.to_dict(orient="records"):
        cohort = row["cohort"]
        df = metrics_by_cohort[cohort]
        metrics_path = metrics_path_for(cohort)
        max_step = int(df["step"].max()) if not df.empty else -1
        rows.append(
            {
                **row,
                "metrics_rows": int(len(df)),
                "task_metric_rows": int(df["task_key"].isin(TASK_KIND).sum())
                if not df.empty
                else 0,
                "max_step": max_step,
                "last_update": pd.Timestamp(metrics_path.stat().st_mtime, unit="s"),
                "state": "complete"
                if max_step >= 4000 and len(df) >= 429
                else f"in progress (step {max_step})",
            }
        )
    return pd.DataFrame(rows)


def build_best_table(df: pd.DataFrame, cohort: str) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    task_df = df[df["task_key"].isin(TASK_KIND)].copy()
    task_df = task_df[task_df["metric_name"] != "loss"]
    rows = []
    for (db_name, table_name), group in task_df.groupby("task_key", sort=False):
        metric_kind = TASK_KIND[(db_name, table_name)]
        val_df = group[group["split"] == "val"].sort_values(
            ["metric_value", "step"], ascending=[False, True]
        )
        test_df = group[group["split"] == "test"].sort_values("step")
        if val_df.empty or test_df.empty:
            continue
        best_val = val_df.iloc[0]
        same_step_test = test_df[test_df["step"] == best_val["step"]]
        best_test = same_step_test.iloc[0] if not same_step_test.empty else test_df.iloc[-1]
        final_test = test_df.iloc[-1]
        rows.append(
            {
                "cohort": cohort,
                "db_name": db_name,
                "table_name": table_name,
                "metric_kind": metric_kind,
                "task": f"{db_name}/{table_name}",
                "best_val_step": int(best_val["step"]),
                "best_val_value": float(best_val["metric_value"]),
                "best_test_step": int(best_test["step"]),
                "best_test_value": float(best_test["metric_value"]),
                "final_test_step": int(final_test["step"]),
                "final_test_value": float(final_test["metric_value"]),
                "final_minus_best": float(final_test["metric_value"] - best_test["metric_value"]),
            }
        )
    return pd.DataFrame(rows)


def plot_delta_bars(delta_df: pd.DataFrame, title: str, xlabel: str) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(15, 9), constrained_layout=True)
    for ax, metric_kind in zip(axes, ["auc", "r2"]):
        sub = delta_df[delta_df["metric_kind"] == metric_kind].sort_values("delta")
        colors = ["#d95f02" if value < 0 else "#1b9e77" for value in sub["delta"]]
        ax.barh(sub["task"], sub["delta"], color=colors)
        ax.axvline(0, color="black", linewidth=1)
        ax.set_title(metric_kind.upper())
        ax.set_xlabel(xlabel)
        ax.invert_yaxis()
    fig.suptitle(title)
    plt.show()


def plot_late_drop_bars(drift_df: pd.DataFrame, title: str) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(15, 8), constrained_layout=True)
    for ax, cohort in zip(axes, ["G0_hsbm", "G7_realistic_mix"]):
        sub = drift_df[drift_df["cohort"] == cohort].sort_values("final_minus_best").head(6)
        colors = ["#d95f02" if value < 0 else "#1b9e77" for value in sub["final_minus_best"]]
        ax.barh(sub["task"], sub["final_minus_best"], color=colors)
        ax.axvline(0, color="black", linewidth=1)
        ax.set_title(cohort)
        ax.set_xlabel("final test - best test")
        ax.invert_yaxis()
    fig.suptitle(title)
    plt.show()


def parse_paper_table(
    html_text: str, heading: str, task_order: list[tuple[str, str]], metric_kind: str
) -> pd.DataFrame:
    start = html_text.index(heading)
    body_start = html_text.index("<tbody>", start)
    body_end = html_text.index("</tbody>", body_start)
    body = html_text[body_start:body_end]
    rows = re.findall(
        r"<tr><td>(.*?)</td><td>(.*?)</td><td>(.*?)</td><td>(.*?)</td><td>(.*?)</td><td[^>]*>(.*?)</td></tr>",
        body,
    )
    if len(rows) != len(task_order):
        raise ValueError(f"Expected {len(task_order)} rows in {heading!r}, found {len(rows)}")
    records = []
    for (db_name, table_name), row in zip(task_order, rows):
        records.append(
            {
                "db_name": db_name,
                "table_name": table_name,
                "metric_kind": metric_kind,
                "paper_real_only": float(html_lib.unescape(row[2])) / 100.0,
                "paper_syn_only": float(html_lib.unescape(row[3])) / 100.0,
                "paper_syn_plus_real": float(html_lib.unescape(row[4])) / 100.0,
                "paper_gain": float(html_lib.unescape(row[5])) / 100.0,
            }
        )
    return pd.DataFrame(records)


def load_topology_series(path: Path) -> pd.Series:
    obj = json.loads(path.read_text(encoding="utf-8"))
    return pd.Series({metric: obj["metric_summary"][metric]["mean"] for metric in TOPO_STATS})


def plot_topology_deltas(delta_df: pd.DataFrame, title: str) -> None:
    small_metrics = [metric for metric in TOPO_STATS if metric != "powerlaw_gamma"]
    fig, ax = plt.subplots(figsize=(13, 5))
    delta_df[small_metrics].T.plot(kind="bar", ax=ax)
    ax.axhline(0, color="black", linewidth=1)
    ax.set_ylabel("synthetic - real")
    ax.set_title(title)
    ax.legend(title="cohort")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()


def plot_powerlaw_gamma(topology_df: pd.DataFrame) -> None:
    fig, ax = plt.subplots(figsize=(8, 4))
    topology_df.loc[["real", "G0_hsbm", "G7_realistic_mix", "G7_db"], ["powerlaw_gamma"]].plot(
        kind="bar", ax=ax
    )
    ax.axhline(
        topology_df.loc["real", "powerlaw_gamma"], color="black", linestyle="--", linewidth=1
    )
    ax.set_ylabel("mean value")
    ax.set_title("Power-law gamma (separate scale)")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

## 1. Pipeline audit

先固定四个最容易出错的点：
- target 是否被 mask。
- 邻居/边是否会跨过时间边界。
- synthetic 评估是不是把 test split 误算进去。
- 最终结果是不是只看 best-val checkpoint，而不是最后一步。

结论先写在前面：
- 没看到直接目标泄露。
- `rel-synthetic-loss//test/loss: nan` 这种现象是预期行为，不是异常。
- 真正需要警惕的是上下文截断和 best-val 选点带来的偏差，而不是明显的数据穿越。


In [ ]:
manifest_df = load_manifest(MANIFEST_PATH)
metrics_by_cohort = {cohort: load_metrics(cohort) for cohort in COHORT_ORDER}
run_status_df = build_run_status(manifest_df, metrics_by_cohort)

display(manifest_df)
display(
    run_status_df[
        [
            "cohort",
            "assignment_strategy",
            "config_hash",
            "metrics_rows",
            "task_metric_rows",
            "max_step",
            "state",
            "last_update",
        ]
    ]
)

In [ ]:
completed_cohorts = ["G0_hsbm", "G7_realistic_mix"]
best_df = pd.concat(
    [build_best_table(metrics_by_cohort[cohort], cohort) for cohort in completed_cohorts],
    ignore_index=True,
)
comparison_df = best_df.pivot_table(
    index=["db_name", "table_name", "metric_kind"],
    columns="cohort",
    values="best_test_value",
).reset_index()
comparison_df["task"] = comparison_df["db_name"] + "/" + comparison_df["table_name"]
comparison_df["delta"] = comparison_df["G7_realistic_mix"] - comparison_df["G0_hsbm"]

display(
    comparison_df[["task", "metric_kind", "G0_hsbm", "G7_realistic_mix", "delta"]].sort_values(
        ["metric_kind", "delta"], ascending=[True, False]
    )
)

delta_summary = (
    comparison_df.groupby("metric_kind", as_index=False)["delta"]
    .agg(["mean", "median", "min", "max"])
    .reset_index()
)
display(delta_summary)

plot_delta_bars(
    comparison_df.rename(columns={"delta": "delta"}),
    title="G7_realistic_mix vs G0_hsbm: best-test metric delta",
    xlabel="G7_realistic_mix - G0_hsbm",
)

In [ ]:
drift_df = best_df.copy()
display(
    drift_df[
        ["cohort", "task", "metric_kind", "best_test_value", "final_test_value", "final_minus_best"]
    ].sort_values("final_minus_best")
)

display(
    drift_df.groupby(["cohort", "metric_kind"], as_index=False)["final_minus_best"]
    .mean()
    .pivot(index="metric_kind", columns="cohort", values="final_minus_best")
)

plot_late_drop_bars(drift_df, title="Late-step regression: final test minus best test")

In [ ]:
html_text = DOCS_HTML.read_text(encoding="utf-8")
paper_df = pd.concat(
    [
        parse_paper_table(html_text, "Classification (AUROC %)", AUC_TASKS, "auc"),
        parse_paper_table(html_text, "Regression (R&sup2; %)", R2_TASKS, "r2"),
    ],
    ignore_index=True,
)

paper_compare_df = best_df.merge(paper_df, on=["db_name", "table_name", "metric_kind"], how="inner")
paper_compare_df["delta_to_paper"] = (
    paper_compare_df["best_test_value"] - paper_compare_df["paper_syn_only"]
)

display(
    paper_compare_df[
        [
            "cohort",
            "task",
            "metric_kind",
            "best_test_value",
            "paper_syn_only",
            "delta_to_paper",
        ]
    ].sort_values(["cohort", "metric_kind", "delta_to_paper"], ascending=[True, True, False])
)

paper_gap_summary = (
    paper_compare_df.groupby(["cohort", "metric_kind"], as_index=False)["delta_to_paper"]
    .agg(["mean", "median", "min", "max"])
    .reset_index()
)
display(paper_gap_summary)

paper_gap_plot = paper_compare_df.groupby(["cohort", "metric_kind"], as_index=False)[
    "delta_to_paper"
].mean()
fig, ax = plt.subplots(figsize=(8, 4))
paper_gap_plot.pivot(index="metric_kind", columns="cohort", values="delta_to_paper").loc[
    ["auc", "r2"]
].plot(kind="bar", ax=ax)
ax.axhline(0, color="black", linewidth=1)
ax.set_ylabel("best-test - paper synthetic-only")
ax.set_title("Gap to paper synthetic-only column")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
topo_paths = {
    "real": REAL_TOPO_PATH,
    "G0_hsbm": RUNTIME_ROOT
    / "experiments/g0_g7_g7db_paired_n34_seed30000/topology_stats.G0_hsbm/summary.json",
    "G7_realistic_mix": RUNTIME_ROOT
    / "experiments/g0_g7_g7db_paired_n34_seed30000/topology_stats.G7_realistic_mix/summary.json",
    "G7_db": RUNTIME_ROOT
    / "experiments/g0_g7_g7db_paired_n34_seed30000/topology_stats.G7_db/summary.json",
}
topology_df = pd.DataFrame(
    {name: load_topology_series(path) for name, path in topo_paths.items()}
).T
topology_delta_df = topology_df.drop(index="real").subtract(topology_df.loc["real"], axis=1)

display(topology_df.round(4))
display(topology_delta_df.round(4))
display(topology_df[["temporal_growth_alpha"]].round(4))

plot_topology_deltas(topology_delta_df, title="Topology deltas vs real benchmark")
plot_powerlaw_gamma(topology_df)

## 结论

- 没看到直接目标泄露：target mask、timestamp cutoff、columns_to_drop 都在代码里显式处理。
- `G7_realistic_mix` 不是全面胜出：AUC 平均略低于 `G0_hsbm`，R2 平均略高，但幅度都很小。
- 一些任务出现明显晚期回落，说明 best-val 选点很重要，不能直接看最后一步。
- 跟论文 synthetic-only 比，我们当前两组都还差一截，尤其是 `driver-position`、`study-outcome`、`item-churn` 这些任务。
- 结构上最可疑的不是泄露，而是表示能力不足：`ctx_len=1024`、`max_bfs_width=128`、`MAX_F2P_NBRS=5` 会截断上下文；`temporal_growth_alpha` 仍固定在 1.0，和 real benchmark 的 1.547 差很大。
- `G7_db` 已经跑完，但它没有把结构问题一次性解决掉，整体仍然更像是局部任务有提升、全局不稳定。

下一步最值得做的事：如果后续继续开新配置，沿着同一套 best-val/test 和 paper synthetic-only 的表格继续补，这样能直接横向比较；如果结果仍然不稳，优先查 `ctx_len` / `max_bfs_width` / `MAX_F2P_NBRS` 这些截断点，再考虑改生成侧的时间增长约束。
